In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# DATA PREPARATION

In [2]:
DATASETS_PATH = "../Datasets/"

In [3]:
income = pd.read_csv(DATASETS_PATH + 'Average personal income.csv', sep=';')
green = pd.read_csv(DATASETS_PATH + 'green-spaces.csv')
pollution = pd.read_csv(DATASETS_PATH + 'merged_air_pollution_data_clean.csv')

In [ ]:
income = income[['Neighbourhood', '2022']].rename(columns={'2023': 'Income_2022'})

In [ ]:
income['Income_2022'] = (
    income['Income_2022']
    .replace('.', np.nan)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

In [ ]:
income = income.dropna(subset=['Income_2022'])

In [7]:
green['NEIGHBOURHOOD'] = green['NEIGHBOURHOOD'].str.strip()
green_grouped = green.groupby('NEIGHBOURHOOD').agg(
    total_green_space=('SURFACE_AREA', 'sum'),
    num_green_areas=('SURFACE_AREA', 'count'),
    avg_green_area=('SURFACE_AREA', 'mean')
).reset_index()

In [8]:
pollution['Neighbourhood'] = pollution['Neighbourhood'].str.strip()
pollution_grouped = pollution.groupby('Neighbourhood').agg({
    'PM2.5_mean': 'mean',
    'PM10_mean': 'mean',
    'NO2_mean': 'mean'
}).reset_index()

In [ ]:
# the giga dataset
df = income.merge(pollution_grouped, left_on='Neighbourhood', right_on='Neighbourhood', how='inner')
df = df.merge(green_grouped, left_on='Neighbourhood', right_on='NEIGHBOURHOOD', how='inner')

In [12]:
df

,Neighbourhood,Income_2023,NEIGHBOURHOOD,total_green_space,num_green_areas,avg_green_area
0,Achtse Barrier-Gunterslaer,41.5,Achtse Barrier-Gunterslaer,141028.292,506,278.712040
1,Achtse Barrier-Hoeven,40.2,Achtse Barrier-Hoeven,132439.353,424,312.356965
2,Achtse Barrier-Spaaihoef,44.0,Achtse Barrier-Spaaihoef,193523.418,390,496.213892
3,Bennekel-Oost,32.8,Bennekel-Oost,179586.744,450,399.081653
4,"Bennekel-West, Gagelbosch",33.3,"Bennekel-West, Gagelbosch",169740.232,451,376.364151
5,Binnenstad,44.2,Binnenstad,37590.173,581,64.699093
6,Blixembosch-Oost,51.6,Blixembosch-Oost,404366.870,1059,381.838404
7,Doornakkers-West,34.6,Doornakkers-West,76659.236,408,187.890284
8,Eckart,30.8,Eckart,185511.429,805,230.448980
9,"Eliasterrein, Vonderkwartier",46.8,"Eliasterrein, Vonderkwartier",28344.669,323,87.754393


In [13]:
df = df.drop(columns=['NEIGHBOURHOOD'])

# MODELLING

In [14]:
features = ['total_green_space', 'num_green_areas', 'avg_green_area', 'PM2.5_mean', 'PM10_mean', 'NO2_mean']
X = df[features]
y = df['Income_2023']

KeyError: "['PM2.5_mean', 'PM10_mean', 'NO2_mean'] not in index"

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)